# 🔬 Fine-Tuning Small Language Models with LoRA / QLoRA
### End-to-End Pipeline: Bilingual Domain QA (English + Hindi) on Google Colab (Free T4 GPU)

This notebook trains and evaluates an open-weight Small Language Model (`Qwen/Qwen2.5-1.5B-Instruct` or `meta-llama/Llama-3.2-1B-Instruct`) adapted for **Bilingual Domain QA (Agriculture & Government Citizen Schemes)** using **4-bit QLoRA**.

#### Key Objectives:
1. **Sanity Check (Step 7):** Rapid overfit verification on ~50 samples.
2. **Full QLoRA Training:** NF4 double quantization with completion-only loss masking.
3. **Baselines & Evaluation:** Compare tuned model vs zero-shot prompted baseline on Token F1, Exact Match, and Latency.
4. **Deployment:** Merge adapter (`merge_and_unload()`) and launch an interactive Gradio demo.

--- 
## 1. Verify GPU Acceleration & Install Dependencies
> **Ensure GPU is selected:** Navigate to **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
# Check GPU availability
!nvidia-smi

# Remove Colab's outdated pre-installed torchao (conflicts with newer PEFT; we use bitsandbytes)
!pip uninstall -y -q torchao

# Install pinned dependencies
!pip install -q --upgrade pip
!pip install -q \
    "torch>=2.2.0" \
    "transformers>=4.42.0" \
    "peft>=0.11.0" \
    "trl>=0.9.0" \
    "bitsandbytes>=0.43.1" \
    "datasets>=2.19.0" \
    "accelerate>=0.30.0" \
    "evaluate>=0.4.1" \
    "sentence-transformers>=2.7.0" \
    "faiss-cpu>=1.8.0" \
    "gradio>=4.31.0" \
    "pyyaml>=6.0.1" \
    "scikit-learn"

--- 
## 2. Clone Repository or Set Up Project Files

In [ ]:
import os, sys
# If running in Colab after cloning your GitHub repo:
# !git clone https://github.com/<your-username>/<your-repo-name>.git repo
# %cd repo

# Ensure project root is in sys.path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

--- 
## 3. Hugging Face & W&B Login (Optional for Public Models)

In [ ]:
# If using gated models like Llama-3.2, enter your Hugging Face User Access Token
from huggingface_hub import login
# login(token="hf_your_token_here")

# If tracking experiments with Weights & Biases:
# import wandb
# wandb.login()

--- 
## 4. Step 7: Sanity Check (Rapid Overfit Run)
First, we overfit on 50 samples to confirm that gradients flow, loss descends rapidly, and output formatting matches the target.

In [ ]:
!python -m src.training.train_qlora --config configs/sanity_check.yaml

--- 
## 5. Main QLoRA Training Run
Trains `Qwen/Qwen2.5-1.5B-Instruct` using **4-bit NF4 double quantization** with LoRA rank $r=16, \alpha=32$ on all linear projection layers.

In [ ]:
# Generate seed leak-free splits if your full curated dataset isn't placed in data/splits/ yet
!python -m src.data.generate_domain_qa_dataset

# Run unit tests to verify data loader, metrics, and configs
!pytest tests/

# Execute main QLoRA training
!python -m src.training.train_qlora --config configs/base_config.yaml

--- 
## 6. Baselines & Rigorous Evaluation
Compare the fine-tuned adapter against the zero-shot base model on the frozen test set.

In [ ]:
from src.data.dataset_loader import load_jsonl_dataset
from src.evaluation.evaluator import ModelEvaluator

test_dataset = load_jsonl_dataset("data/splits/test.jsonl")
test_records = [item for item in test_dataset]

print(f"Evaluating {len(test_records)} frozen test samples...")

# 1. Evaluate Zero-shot Base Model
base_evaluator = ModelEvaluator(base_model_id="Qwen/Qwen2.5-1.5B-Instruct")
base_results = base_evaluator.evaluate_test_set(
    test_records,
    task_type="qa",
    output_report_path="results/metrics/base_model_results.json"
)
print("Base Model Benchmark:", base_results)
del base_evaluator
import gc, torch; gc.collect(); torch.cuda.empty_cache()

# 2. Evaluate LoRA Fine-Tuned Model
lora_evaluator = ModelEvaluator(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    adapter_path="outputs/qlora_run/adapter"
)
lora_results = lora_evaluator.evaluate_test_set(
    test_records,
    task_type="qa",
    output_report_path="results/metrics/qlora_results.json"
)
print("Fine-Tuned Model Benchmark:", lora_results)

--- 
## 7. Latency, Throughput & Peak VRAM Benchmark (RQ5)

In [ ]:
from src.evaluation.benchmark_latency import benchmark_inference_efficiency

latency_stats = benchmark_inference_efficiency(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    adapter_path="outputs/qlora_run/adapter",
    num_runs=15
)
print("Efficiency Benchmark:", latency_stats)

--- 
## 8. Merge LoRA Adapter to 16-Bit Standalone Checkpoint

In [ ]:
from src.serving.merge_adapters import merge_lora_to_standalone

merged_path = merge_lora_to_standalone(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    adapter_dir="outputs/qlora_run/adapter",
    output_dir="outputs/merged_qwen_1.5b_domain_qa"
)
print(f"Model merged and ready at: {merged_path}")

--- 
## 9. Launch Live Interactive Comparison Demo (Gradio)

In [ ]:
# Free VRAM from earlier evaluation cells before launching Gradio
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Launches interactive demo with a public shareable gradio.live URL
!python src/serving/app.py --share